In [ ]:
#Signal Processing and Frequency Analysis
#This notebook processes CSV files obtained from the vibration sensor
#acquisition code. It detects signal peaks, calculates the vibration
#frequency for each sample, and stores the mean results in a CSV file.

import pandas as pd
import numpy as np
from scipy.signal import find_peaks
import matplotlib.pyplot as plt


# 1. Configuration
csv_files = [
    r"C:\Users\inese\OneDrive\Documentos\1. 4º ING\TFG\Código Python\DEF3_SM_3.3V_muestra1.csv",
    r"C:\Users\inese\OneDrive\Documentos\1. 4º ING\TFG\Código Python\DEF3_SM_3.3V_muestra2.csv",
    r"C:\Users\inese\OneDrive\Documentos\1. 4º ING\TFG\Código Python\DEF3_SM_3.3V_muestra3.csv"
]

motor = "Small_motor"       # Change according to the analysed motor
input_voltage = 3.3         # Change according to the analysed voltage

frequencies = []            # Lists used to store the results
cycles = []


# 2. Analysis of each CSV File
for csv_file in csv_files:
    data = pd.read_csv(csv_file, sep=";", decimal=",")
    t = data["time_s"].values
    v = data["voltage_V"].values
    dt = np.mean(np.diff(t))  # Sampling interval

    # Frequency calculation, with the function find_peaks()
    min_distance = int(0.006 / dt)                                       # Minimum distance between consecutive peaks (6 ms)
    peaks, _ = find_peaks(v, prominence=0.004, distance=min_distance)    # Detect signal peaks using prominence and minimum distance criteria
    t_peaks = t[peaks]                                                   # Extract the time corresponding to each detected peak
    periods = np.diff(t_peaks)                                           # Calculate the period between consecutive peaks
    T = np.mean(periods)                                                 
    frequency = 1 / T

    # Number of cycles analysed
    num_cycles = len(periods)

    # Store individual results
    frequencies.append(frequency)
    cycles.append(num_cycles)


#3. Final mean values and standard deviations 
mean_frequency = np.mean(frequencies)
std_frequency = np.std(frequencies)
mean_cycles = np.mean(cycles)


# 4. Display results
print("RESULTS:")
print(f"Frequency = {mean_frequency:.2f} Hz ± {std_frequency:.2f} Hz")
print(f"Cycles analysed = {mean_cycles:.0f}")


# 5. Save results 
mean_results = pd.DataFrame([{
    "Motor": motor,
    "Input_Voltage": input_voltage,
    "Number_of_samples": len(csv_files),
    "Frequency_Hz_mean": mean_frequency,
    "Frequency_std_Hz": std_frequency,
    "Cycles_analysed_mean": mean_cycles
}])

output_file = "frequency_results_SM.csv"
mean_results.to_csv(
    output_file,
    mode="a",
    header=not pd.io.common.file_exists(output_file),
    index=False,
    sep=";",
    decimal=","
)


# 6. Plot signal with detected peaks
plt.figure(figsize=(10, 5))
plt.plot(t, v, label="Signal")
plt.plot(t_peaks, v[peaks], "rx", label="Detected peaks")  # Peaks used to calculate frequency
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.title(f"{motor} | {input_voltage} V")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
